# 91 — SmolVLA tree-Q10 training, worker 1: return + stock-relative gap

Trains the same RL-token Q10 critic on the same frozen 521 complete depth-1 trees. In addition to the decaying terminal-return target, this arm adds a Huber loss matching each candidate-minus-stock predicted Q gap to its observed return gap, with coefficient 1.0.

The cache, split, initialization, sampled tree batches, optimizer, and schedule match worker 0 exactly. Verify that both notebooks print identical dataset and initial-model digests.

In [ ]:
EXTRAS = 'train'
SETUP_ENV = False
import urllib.request
exec(urllib.request.urlopen(
    'https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/'
    'pnp-vla/scripts/colab_bootstrap.py').read().decode())

In [ ]:
from google.colab import drive
import shutil
drive.mount('/content/drive')

RUN_NAME = 'return_plus_stock_gap'
DIFFERENCE_WEIGHT = 1.0
EXPECTED_TREES = 521
UPDATES = 2000
MICRO_TREE_BATCH = 2  # lower to 1 only if this GPU runs out of memory
DOWNLOAD_WORKERS = 8
CACHE_ROOT = '/content/smolvla_tree_q10_cache'
OUTPUT_ROOT = '/content/drive/MyDrive/pnp_smolvla_tree_q10'

disk = shutil.disk_usage('/content')
print({
    'experiment': 'smolvla-tree-q10-return-gap-ablation-v1',
    'arm': RUN_NAME, 'difference_weight': DIFFERENCE_WEIGHT,
    'expected_frozen_trees': EXPECTED_TREES, 'updates': UPDATES,
    'gamma_per_environment_action': 0.99,
    'executed_action_horizon': 10, 'generated_chunk_size': 50,
    'effective_tree_batch': 8, 'effective_candidate_rows': 72,
    'micro_tree_batch': MICRO_TREE_BATCH,
    'print_every': 100, 'validate_every': 250, 'checkpoint_every': 500,
    'local_free_GiB': round(disk.free / 2**30, 1),
    'cache_root': CACHE_ROOT, 'output_root': OUTPUT_ROOT,
})

In [ ]:
from pnp.smolvla_tree_critic import run_smolvla_tree_q10_training

report = run_smolvla_tree_q10_training(
    run_name=RUN_NAME, difference_weight=DIFFERENCE_WEIGHT,
    expected_trees=EXPECTED_TREES, updates=UPDATES,
    cache_root=CACHE_ROOT, output_root=OUTPUT_ROOT,
    micro_tree_batch=MICRO_TREE_BATCH, download_workers=DOWNLOAD_WORKERS,
    resume=True)
report